In [1]:
import os
import sys

from matplotlib import pyplot as plt
import mlflow
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import seaborn as sns
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join('..')))
from models import GNNModel
from torch_geometric.loader import DataLoader
from utils import mol_to_graph
from utils import MLFlowManager


from joblib import Parallel, delayed

def log_regression_plots(y_true, y_pred, run_name):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], '--r', lw=2)
    plt.xlabel("Actual pIC50")
    plt.ylabel("Predicted pIC50")
    plt.title(f"Regression Fit - {run_name}")
    
    plot_path = "pred_vs_actual.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()

parquet_path = "parquets/df_ml_with_scaffold.parquet"
df = pd.read_parquet(parquet_path)

print("SMILES to graph conversion...")
dataset_path = "/home/pkuszn/repos/WSzI/src/notebooks/data/chembl_dataset.pt"

if os.path.exists(dataset_path):
    print("Loading existing dataset from disk...")
    dataset = torch.load(dataset_path, weights_only=False)
    print(f"Loaded {len(dataset)} graphs.")
else:
    print("Dataset not found. Starting SMILES to graph conversion...")
    smiles_list = df['canonical_smiles'].tolist()
    pic50_list = df['pic50'].tolist()
    
    dataset = Parallel(n_jobs=-1)(
        delayed(mol_to_graph)(s, y) 
        for s, y in tqdm(zip(smiles_list, pic50_list), total=len(df), desc="Converting")
    )
    dataset = [d for d in dataset if d is not None] 
    
    torch.save(dataset, dataset_path)
    print(f"Conversion complete. Saved {len(dataset)} graphs to disk.")

mf = MLFlowManager(experiment_name="ChEMBL_GNN_Scaffold_Split")

model_types = ["GCN", "GIN"]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)
# train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
smoke_test_loader = DataLoader(train_data[:64], batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
for m_type in model_types:
    model = GNNModel(num_node_features=4, hidden_channels=64, model_type=m_type).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        
    run_name = f"Run_{m_type}_ScaffoldData"
    print(f"Starting: {run_name}")
    
    model.train_gnn(
        model=model, 
        loader=smoke_test_loader, 
        optimizer=optimizer, 
        device=device, 
        mf_manager=mf, 
        run_name=run_name
    )

    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for data in smoke_test_loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch).view(-1)
            y_true.extend(data.y.cpu().numpy())
            y_pred.extend(out.cpu().numpy())

    model_save_path = f"model_{m_type}_weights.pth"
    torch.save(model.state_dict(), model_save_path)
    print(f"Saved {m_type} weights to {model_save_path}")
    
    final_r2 = r2_score(y_true, y_pred)
    final_mae = mean_absolute_error(y_true, y_pred)

    with mlflow.start_run(run_name=run_name, nested=True):
        mlflow.log_metric("final_test_r2", final_r2)
        mlflow.log_metric("final_test_mae", final_mae)
        
        log_regression_plots(y_true, y_pred, m_type)

    print(f"Finished {m_type}: R2={final_r2:.4f}, MAE={final_mae:.4f}")


SMILES to graph conversion...
Loading existing dataset from disk...
Loaded 1488023 graphs.
Starting: Run_GCN_ScaffoldData


2026/06/04 13:20:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 13:20:06 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Finished Run_GCN_ScaffoldData with Loss: 108.1244
🏃 View run Run_GCN_ScaffoldData at: http://localhost:5000/#/experiments/4/runs/22f9328738894085871812333e49c23c
🧪 View experiment at: http://localhost:5000/#/experiments/4
Saved GCN weights to model_GCN_weights.pth


2026/06/04 13:20:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Run_GCN_ScaffoldData at: http://localhost:5000/#/experiments/4/runs/a53c6d654cea4a758bd7fe307c05bcc6
🧪 View experiment at: http://localhost:5000/#/experiments/4
Finished GCN: R2=-1.9047, MAE=1.7112
Starting: Run_GIN_ScaffoldData


2026/06/04 13:20:14 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Finished Run_GIN_ScaffoldData with Loss: 554.5857
🏃 View run Run_GIN_ScaffoldData at: http://localhost:5000/#/experiments/4/runs/68042c14df664f479bac8bcef147df88
🧪 View experiment at: http://localhost:5000/#/experiments/4
Saved GIN weights to model_GIN_weights.pth
🏃 View run Run_GIN_ScaffoldData at: http://localhost:5000/#/experiments/4/runs/7849d430368447f7a5787b941cf034b8
🧪 View experiment at: http://localhost:5000/#/experiments/4
Finished GIN: R2=-87.2490, MAE=11.7329
